# Non-parametric regression with k-nearest neighbors

## Initialize

In [ ]:
# import required libraries
import numpy as np
from numpy.polynomial.polynomial import Polynomial
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import tensorflow as tf #type: ignore
import torch #type: ignore
#
from IPython.display import display, Math
# Import typyng library for type hinting
from typing import Callable, Union, List, Any, Tuple
#
# Initialize pretty printing
sp.init_printing()

## 2NN (polynomisl data)

In [ ]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def design_matrix(degree: int, 
                  x_values: np.ndarray
                 ) -> np.ndarray:
    """Generates a design matrix for polynomial regression of a specified degree.
    
    Args:
        degree: The degree of the polynomial hypothesis.
        x_values: A numpy array of x values.
    
    Returns:
        A design matrix as a numpy array.
    """
    return np.array([[x**i for i in range(degree+1)] for x in x_values])

def omega_solution(X: np.ndarray, 
                   Y: np.ndarray
                  ) -> np.ndarray:
    """Calculates the polynomial coefficients (omega) that best fit the data.
    
    Args:
        X: The design matrix.
        Y: A numpy array of target y values.
    
    Returns:
        A numpy array of calculated omega values.
    """
    return np.linalg.pinv(X.T @ X) @ X.T @ Y

def poly_solution(omega_solution: np.ndarray) -> Polynomial:
    """Converts omega coefficients to a numpy Polynomial object.
    
    Args:
        omega_solution: A numpy array of omega coefficients.
    
    Returns:
        A numpy Polynomial object representing the fitted polynomial.
    """
    return Polynomial(omega_solution)

def MSE(omega_solution: np.ndarray, 
        x_values: np.ndarray,
        y_true: np.ndarray
       ) -> float:
    """Calculates the Mean Squared Error (MSE) for given omega coefficients and x values.
    
    Args:
        omega_solution: A numpy array of omega coefficients.
        x_values: A numpy array of x values for evaluation.
        y_true: A numpy array of true y values.
    
    Returns:
        The MSE as a float.
    """
    poly: Polynomial = poly_solution(omega_solution)
    y_pred: np.ndarray = poly(x_values)
    return np.mean((y_true - y_pred)**2)

In [ ]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3, 4, 1)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)
train_data

array([[-3.        ,  2.87523453],
       [-2.        ,  3.00934707],
       [-1.        ,  2.7780358 ],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.27298745],
       [ 2.        ,  4.51421884],
       [ 3.        ,  1.84617967]])

In [ ]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[1:3]])] for x in x_train])
pred_val = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[0:2]])] for x in x_val])
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.89369144]
 [-2.5         2.9422908 ]
 [-2.          2.82663516]
 [-1.5         2.89369144]
 [-1.          2.37845552]
 [-0.5         2.26279988]
 [ 0.          3.52551163]
 [ 0.5         3.01027571]
 [ 1.          3.1308914 ]
 [ 1.5         4.39360315]
 [ 2.          3.05958356]
 [ 2.5         3.18019926]
 [ 3.          4.39360315]]


In [ ]:
train_loss = np.mean((y_train - pred_train[:, 1])**2)
val_loss = np.mean((y_val - pred_val[:, 1])**2)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


## 2NN (non-polynomisl data)

In [ ]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return 3 + np.sin(x) - np.cos(x)

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

In [ ]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3, 4, 1)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)
train_data

array([[-3.        ,  2.09910702],
       [-2.        ,  2.84952981],
       [-1.        ,  2.77126251],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.28248947],
       [ 2.        ,  4.8396631 ],
       [ 3.        ,  4.35229217]])

In [ ]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[1:3]])] for x in x_train])
pred_val = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[0:2]])] for x in x_val])
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.81039616]
 [-2.5         2.47431841]
 [-2.          2.43518476]
 [-1.5         2.81039616]
 [-1.          2.29854689]
 [-0.5         2.25941324]
 [ 0.          3.52687599]
 [ 0.5         3.01502671]
 [ 1.          3.29361353]
 [ 1.5         4.56107629]
 [ 2.          4.31739082]
 [ 2.5         4.59597764]
 [ 3.          4.56107629]]


In [ ]:
train_loss = np.mean((y_train - pred_train[:, 1])**2)
val_loss = np.mean((y_val - pred_val[:, 1])**2)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.766, Validation loss: 0.466


## Generalized 2NN (polynomisl data)

In [165]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def design_matrix(degree: int, 
                  x_values: np.ndarray
                 ) -> np.ndarray:
    """Generates a design matrix for polynomial regression of a specified degree.
    
    Args:
        degree: The degree of the polynomial hypothesis.
        x_values: A numpy array of x values.
    
    Returns:
        A design matrix as a numpy array.
    """
    return np.array([[x**i for i in range(degree+1)] for x in x_values])

def omega_solution(X: np.ndarray, 
                   Y: np.ndarray
                  ) -> np.ndarray:
    """Calculates the polynomial coefficients (omega) that best fit the data.
    
    Args:
        X: The design matrix.
        Y: A numpy array of target y values.
    
    Returns:
        A numpy array of calculated omega values.
    """
    return np.linalg.pinv(X.T @ X) @ X.T @ Y

def poly_solution(omega_solution: np.ndarray) -> Polynomial:
    """Converts omega coefficients to a numpy Polynomial object.
    
    Args:
        omega_solution: A numpy array of omega coefficients.
    
    Returns:
        A numpy Polynomial object representing the fitted polynomial.
    """
    return Polynomial(omega_solution)

def MSE(omega_solution: np.ndarray, 
        x_values: np.ndarray,
        y_true: np.ndarray
       ) -> float:
    """Calculates the Mean Squared Error (MSE) for given omega coefficients and x values.
    
    Args:
        omega_solution: A numpy array of omega coefficients.
        x_values: A numpy array of x values for evaluation.
        y_true: A numpy array of true y values.
    
    Returns:
        The MSE as a float.
    """
    poly: Polynomial = poly_solution(omega_solution)
    y_pred: np.ndarray = poly(x_values)
    return np.mean((y_true - y_pred)**2)

In [166]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)
train_data

array([[-3.        ,  2.87523453],
       [-2.        ,  3.00934707],
       [-1.        ,  2.7780358 ],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.27298745],
       [ 2.        ,  4.51421884],
       [ 3.        ,  1.84617967]])

In [167]:
def pred_func(x_data: np.ndarray,
              train_data: np.ndarray,
              m: float = 1.0,
              n: float = 1.0
             ) -> np.ndarray:
    """Computes the training predictions for the given data.
    """
    pred: np.ndarray
    x_train: np.ndarray = train_data[:, 0]
    y_train: np.ndarray = train_data[:, 1]
    #print(f"Training x data: {x_train}")
    #print(f"Input x data: {x_data}")
    if x_data.shape == x_train.shape:
        if np.any(x_data == x_train):
            #print("The input data is the same as the training data.")
            pred = np.array([[x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[1:3]])**n)**(1/m)] for x in x_data])
            #pred = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[1:3]])] for x in x_train])
        else:
            #print("The input data is different from the training data.")
            pred = np.array([[x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[0:2]])**n)**(1/m)] for x in x_data])
            #pred = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[0:2]])] for x in x_data])
    else:
        #print("The input data is different from the training data.")
        pred = np.array([[x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[0:2]])**n)**(1/m)] for x in x_data])
        #pred = np.array([[x, np.mean(y_train[np.argsort(np.abs(x_train - x))[0:2]])] for x in x_data])
    return pred

In [168]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.89369144]
 [-2.5         2.9422908 ]
 [-2.          2.82663516]
 [-1.5         2.89369144]
 [-1.          2.37845552]
 [-0.5         2.26279988]
 [ 0.          3.52551163]
 [ 0.5         3.01027571]
 [ 1.          3.1308914 ]
 [ 1.5         4.39360315]
 [ 2.          3.05958356]
 [ 2.5         3.18019926]
 [ 3.          4.39360315]]


In [169]:
def loss_func(y_true: np.ndarray,
              x_data: np.ndarray,
              train_data: np.ndarray,
              m: float = 1.0,
              n: float = 1.0
             ) -> float:
    """Computes the loss for the given data.
    """
    pred = pred_func(x_data, train_data, m, n)
    return np.mean((y_true - pred[:, 1])**2)

In [170]:
train_loss = loss_func(y_train, x_train, train_data)
val_loss = loss_func(y_val, x_val, train_data)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


### Optimizing loss function over train data

In [171]:
def objective_train(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(y_train, x_train, train_data, m, n)
print(objective_train([1.0, 1.0]))
print(objective_train([0.5, 2.0]))
print(objective_train([2.0, 0.5]))

1.8948857380468203
118937.15251614539
5.310191258361994


In [150]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_train = minimize(objective_train, initial_params, method='Powell', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_train.x
train_loss_opt = loss_func(y_train, x_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(y_val, x_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

Optimal parameters on training set: [ 0.24395654 -0.23740719]
Train loss: 0.6813074745247688
Validation loss: 2.123088412770937


### Optimizing loss function over validation data

In [151]:
def objective_val(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(y_val, x_val, train_data, m, n)
print(objective_val([1.0, 1.0]))
print(objective_val([0.5, 2.0]))
print(objective_val([2.0, 0.5]))

0.40437371232142216
120663.6629126066
3.343110026002987


In [152]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_val = minimize(objective_val, initial_params, method='Powell', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_val.x
train_loss_opt = loss_func(y_train, x_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(y_val, x_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

Optimal parameters on training set: [1.64756999 1.76917139]
Train loss: 1.7087540257863292
Validation loss: 0.1578040758316644


## Generalized 2NN (polynomisl data) - Large Sample

In [190]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 0.1)
    
    return noisy_y_values

def design_matrix(degree: int, 
                  x_values: np.ndarray
                 ) -> np.ndarray:
    """Generates a design matrix for polynomial regression of a specified degree.
    
    Args:
        degree: The degree of the polynomial hypothesis.
        x_values: A numpy array of x values.
    
    Returns:
        A design matrix as a numpy array.
    """
    return np.array([[x**i for i in range(degree+1)] for x in x_values])

def omega_solution(X: np.ndarray, 
                   Y: np.ndarray
                  ) -> np.ndarray:
    """Calculates the polynomial coefficients (omega) that best fit the data.
    
    Args:
        X: The design matrix.
        Y: A numpy array of target y values.
    
    Returns:
        A numpy array of calculated omega values.
    """
    return np.linalg.pinv(X.T @ X) @ X.T @ Y

def poly_solution(omega_solution: np.ndarray) -> Polynomial:
    """Converts omega coefficients to a numpy Polynomial object.
    
    Args:
        omega_solution: A numpy array of omega coefficients.
    
    Returns:
        A numpy Polynomial object representing the fitted polynomial.
    """
    return Polynomial(omega_solution)

def MSE(omega_solution: np.ndarray, 
        x_values: np.ndarray,
        y_true: np.ndarray
       ) -> float:
    """Calculates the Mean Squared Error (MSE) for given omega coefficients and x values.
    
    Args:
        omega_solution: A numpy array of omega coefficients.
        x_values: A numpy array of x values for evaluation.
        y_true: A numpy array of true y values.
    
    Returns:
        The MSE as a float.
    """
    poly: Polynomial = poly_solution(omega_solution)
    y_pred: np.ndarray = poly(x_values)
    return np.mean((y_true - y_pred)**2)

In [197]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 3.1, 0.1)
x_val: np.ndarray = np.arange(-2.95, 2.75, 0.2)
x_test: np.ndarray = np.arange(-2.85, 2.85, 0.2)
x_true: np.ndarray = np.arange(-3., 3.1, 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_test = y_stat(x_test)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
test_data = np.concatenate((x_test.reshape(-1, 1), y_test.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

In [198]:
print(x_train[0:10])
print(x_val[0:10])
print(x_test[0:10])
print(x_train[0:10]-x_val[0:10])
print(x_train[0:10]-x_test[0:10])
print(x_val[0:10]-x_test[0:10])

[-3.  -2.9 -2.8 -2.7 -2.6 -2.5 -2.4 -2.3 -2.2 -2.1]
[-2.95 -2.75 -2.55 -2.35 -2.15 -1.95 -1.75 -1.55 -1.35 -1.15]
[-2.85 -2.65 -2.45 -2.25 -2.05 -1.85 -1.65 -1.45 -1.25 -1.05]
[-0.05 -0.15 -0.25 -0.35 -0.45 -0.55 -0.65 -0.75 -0.85 -0.95]
[-0.15 -0.25 -0.35 -0.45 -0.55 -0.65 -0.75 -0.85 -0.95 -1.05]
[-0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1]


In [199]:
def pred_func(x_data: np.ndarray,
              train_data: np.ndarray,
              m: float = 1.0,
              n: float = 1.0
             ) -> np.ndarray:
    """Computes the training predictions for the given data.
    """
    pred: np.ndarray
    x_train: np.ndarray = train_data[:, 0]
    y_train: np.ndarray = train_data[:, 1]
    if x_data.shape == x_train.shape:
        if np.any(x_data == x_train):
            pred = np.array([[x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[1:3]])**n)**(1/m)] for x in x_data])
        else:
            pred = np.array([[x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[0:2]])**n)**(1/m)] for x in x_data])
    else:
        pred = np.array([[x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[0:2]])**n)**(1/m)] for x in x_data])
    return pred

In [200]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]

In [201]:
def loss_func(y_true: np.ndarray,
              x_data: np.ndarray,
              train_data: np.ndarray,
              m: float = 1.0,
              n: float = 1.0
             ) -> float:
    """Computes the loss for the given data.
    """
    pred = pred_func(x_data, train_data, m, n)
    return np.mean((y_true - pred[:, 1])**2)

In [202]:
train_loss = loss_func(y_train, x_train, train_data)
val_loss = loss_func(y_val, x_val, train_data)
test_loss = loss_func(y_test, x_test, train_data)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}, Test loss: {test_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011, Test loss: 0.021


### Optimizing loss function over train data

In [203]:
def objective_train(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(y_train, x_train, train_data, m, n)
print(objective_train([1.0, 1.0]))
print(objective_train([0.99, 0.99]))
print(objective_train([1.1, 1.1]))
print(objective_train([0.9, 1.1]))
print(objective_train([1.1, 0.9]))

0.020867268685917803
0.021405212086726827
0.05233510375686124
1.4930417376261915
0.5353353627705292


In [204]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_train = minimize(objective_train, initial_params, method='Nelder-Mead', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_train.x
train_loss_opt = loss_func(y_train, x_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(y_val, x_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

# Now, evaluate the test loss using these optimal parameters:
test_loss_opt = loss_func(y_test, x_test, train_data, *optimal_params)
print("Test loss:", test_loss_opt)

Optimal parameters on training set: [1.03507632 1.05502008]
Train loss: 0.020633723279625395
Validation loss: 0.010612547148087608
Test loss: 0.01961604621461617


/var/folders/8p/lb_zz6bs7g7cdkt7y_yjy0z40000gn/T/ipykernel_86164/3951362979.py:5: OptimizeWarning: Unknown solver options: xtol, ftol
  result_train = minimize(objective_train, initial_params, method='Nelder-Mead', options={'xtol': 1e-8, 'ftol': 1e-8})


### Optimizing loss function over validation data

In [206]:
def objective_val(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(y_val, x_val, train_data, m, n)
print(objective_val([1.0, 1.0]))
print(objective_val([0.99, 0.99]))
print(objective_val([1.1, 1.1]))
print(objective_val([0.9, 1.1]))
print(objective_val([1.1, 0.9]))

0.011172250934924315
0.011402588044857349
0.04641712641272331
1.5276135052922213
0.5609926496973789


In [207]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_val = minimize(objective_val, initial_params, method='Powell', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_val.x
train_loss_opt = loss_func(y_train, x_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(y_val, x_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

# Now, evaluate the test loss using these optimal parameters:
test_loss_opt = loss_func(y_test, x_test, train_data, *optimal_params)
print("Test loss:", test_loss_opt)

Optimal parameters on training set: [1.05843285 1.09415469]
Train loss: 0.02080467704398237
Validation loss: 0.010429915552680028
Test loss: 0.01948174476845858
